In [96]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor
from IPython.display import display

model_df = pd.read_csv("../data/processed/model_dataset_final.csv")
model_df["date"] = pd.to_datetime(model_df["date"])

train = model_df[model_df["date"] < "2023-01-01"]
val   = model_df[(model_df["date"] >= "2023-01-01") & (model_df["date"] < "2024-01-01")]
test  = model_df[model_df["date"] >= "2024-01-01"]

print(f"Train: {train.shape}  |  {train['date'].min().date()} → {train['date'].max().date()}")
print(f"Val:   {val.shape}    |  {val['date'].min().date()} → {val['date'].max().date()}")
print(f"Test:  {test.shape}   |  {test['date'].min().date()} → {test['date'].max().date()}")

train (1308793, 13)
val (145917, 13)
test (349947, 13)
train start and end 2012-07-22 00:00:00 2022-12-31 00:00:00
val start and end 2023-01-01 00:00:00 2023-12-31 00:00:00
test start and end 2024-01-01 00:00:00 2026-05-24 00:00:00


In [97]:
FEATURES = [
    "position",             # 0=Attack, 1=Defender, 2=Goalkeeper, 3=Midfield
    "market_value_in_eur",
    "avg_fp_last_3",
    "avg_fp_last_5",
    "avg_goals_last_5",
    "avg_assists_last_5",
    "avg_minutes_last_5",
    "std_fp_last_5",
    "matches_played_last_5"
]

TARGET = "fantasy_points"

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = val[FEATURES]
y_val = val[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

(1308793, 9)
(1308793,)
(145917, 9)
(145917,)
(349947, 9)
(349947,)


In [99]:
lr = LinearRegression()

lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](9,)","[0. ,0. ,0. ,...,0.02,0. ,0. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](9,)","['position','market_value_in_eur','avg_fp_last_3',...,'avg_minutes_last_5', 'std_fp_last_5','matches_played_last_5']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.438
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,9
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,2


### Linear Regression

In [101]:
mae = mean_absolute_error(y_val, val_preds)

rmse = np.sqrt(
    mean_squared_error(y_val, val_preds)
)

r2 = r2_score(y_val, val_preds)

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

MAE : 2.0204
RMSE: 2.6805
R²  : 0.0245


In [102]:
coef_df = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": lr.coef_
})

coef_df.sort_values(
    "coefficient",
    ascending=False
)

,feature,coefficient
6,avg_minutes_last_5,1.542313e-02
3,avg_fp_last_5,4.133014e-04
2,avg_fp_last_3,4.018318e-04
7,std_fp_last_5,3.446967e-04
0,position,8.793072e-05
8,matches_played_last_5,1.743897e-05
5,avg_assists_last_5,7.017190e-06
4,avg_goals_last_5,6.695063e-06
1,market_value_in_eur,1.638767e-08


### Dummy Baseline Comparison

In [103]:
print("y val mean" , y_val.mean())
print("y val std" , y_val.std())
baseline_pred = np.full(
    len(y_val),
    y_train.mean()
)

baseline_mae = mean_absolute_error(
    y_val,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    y_val,
    baseline_pred
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

y val mean 2.4081498386068794
y val std 2.713980731867754
Baseline MAE: 2.0742238595912257
Baseline RMSE: 2.719774859942601
Baseline R²: -0.0042812777252574374


### Random Forest

In [104]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [105]:
rf_val_preds = rf.predict(X_val)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(y_val, rf_val_preds)

rmse = np.sqrt(
    mean_squared_error(y_val, rf_val_preds)
)

r2 = r2_score(y_val, rf_val_preds)

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

MAE : 1.989
RMSE: 2.6467
R²  : 0.0489


In [106]:
feature_importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf.feature_importances_
})

feature_importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
3,avg_fp_last_5,0.561885
6,avg_minutes_last_5,0.139349
1,market_value_in_eur,0.097842
0,position,0.096954
7,std_fp_last_5,0.042288
2,avg_fp_last_3,0.032507
4,avg_goals_last_5,0.017517
5,avg_assists_last_5,0.009333
8,matches_played_last_5,0.002326


### XGBoost (Exploration)

In [111]:
train_sample = train.sample(
    n=1300000,
    random_state=42
)
X_train_sample = train_sample[FEATURES]
y_train_sample = train_sample[TARGET]

In [112]:
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train_sample,
    y_train_sample
)
xgb_val_preds = xgb.predict(X_val)

In [113]:
mae = mean_absolute_error(y_val, xgb_val_preds)

rmse = np.sqrt(
    mean_squared_error(y_val, xgb_val_preds)
)

r2 = r2_score(y_val, xgb_val_preds)

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

MAE : 1.9877
RMSE: 2.6469
R²  : 0.0488


In [114]:
feature_importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": xgb.feature_importances_
})

feature_importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
3,avg_fp_last_5,0.561885
6,avg_minutes_last_5,0.139349
1,market_value_in_eur,0.097842
0,position,0.096954
7,std_fp_last_5,0.042288
2,avg_fp_last_3,0.032507
4,avg_goals_last_5,0.017517
5,avg_assists_last_5,0.009333
8,matches_played_last_5,0.002326


## Final Model — V2 Feature Set (Transfermarkt + Understat)

In [119]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

print("=" * 60)
print("🚀 NOTEBOOK 05: BASELINE MODELING INITIATED")
print("=" * 60)

# 1. Load the freshly generated V2 splits
print("Step 1: Loading Train and Validation sets...")
processed_dir = Path("../data/processed")
train = pd.read_csv(processed_dir / "train.csv")
val = pd.read_csv(processed_dir / "val.csv")

# 2. Separate target and drop structural keys
print("Step 2: Preparing X and y matrices...")
target = "fantasy_points"
structural_cols = ["player_id", "game_id", "date"]

X_train = train.drop(columns=structural_cols + [target])
y_train = train[target]

X_val = val.drop(columns=structural_cols + [target])
y_val = val[target]

print(f"  Training shape:   X={X_train.shape}, y={y_train.shape}")
print(f"  Validation shape: X={X_val.shape}, y={y_val.shape}")

# 3. Establish the "Dumb" Baseline (Predicting the mean every time)
print("\nStep 3: Training Dummy Regressor (Mean Baseline)...")
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

dummy_preds = dummy.predict(X_val)
dummy_rmse = root_mean_squared_error(y_val, dummy_preds)
dummy_mae = mean_absolute_error(y_val, dummy_preds)
dummy_r2 = r2_score(y_val, dummy_preds)

print(f"  🤡 Dummy RMSE: {dummy_rmse:.4f}")
print(f"  🤡 Dummy MAE:  {dummy_mae:.4f}")
print(f"  🤡 Dummy R²:  {dummy_r2:.4f}")

# 4. Establish the ML Baseline (HistGradientBoosting)
print("\nStep 4: Training HistGradientBoostingRegressor Baseline...")
# HGBR is chosen because it handles unscaled data and nulls/zeros natively and is blazing fast
hgbr = HistGradientBoostingRegressor(
    max_iter=100, 
    learning_rate=0.1, 
    random_state=42
)
hgbr.fit(X_train, y_train)

hgbr_preds = hgbr.predict(X_val)
hgbr_rmse = root_mean_squared_error(y_val, hgbr_preds)
hgbr_mae = mean_absolute_error(y_val, hgbr_preds)
hgbr_r2 = r2_score(y_val, hgbr_preds)

print(f"  🤖 HGBR RMSE: {hgbr_rmse:.4f}")
print(f"  🤖 HGBR MAE:  {hgbr_mae:.4f}")
print(f"  🤖 HGBR R²:  {hgbr_r2:.4f}")

🚀 NOTEBOOK 05: BASELINE MODELING INITIATED
Step 1: Loading Train and Validation sets...
Step 2: Preparing X and y matrices...
  Training shape:   X=(1308793, 17), y=(1308793,)
  Validation shape: X=(145917, 17), y=(145917,)

Step 3: Training Dummy Regressor (Mean Baseline)...
  🤡 Dummy RMSE: 2.7198
  🤡 Dummy MAE:  2.0742
  🤡 Dummy R²:  -0.0043

Step 4: Training HistGradientBoostingRegressor Baseline...
  🤖 HGBR RMSE: 2.6441
  🤖 HGBR MAE:  1.9824
  🤖 HGBR R²:  0.0508


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

print("=" * 60)
print("🌲 TRAINING ADVANCED MODELS (RF & XGBOOST)")
print("=" * 60)

# 1. Create a training sample for XGBoost (Matching your previous structure)
print("Step 1: Creating training sample for XGBoost...")
sample_size = min(1300000, len(X_train)) # Limits to 300k rows for speed, adjust as needed
sample_idx = X_train.sample(n=sample_size, random_state=42).index

X_train_sample = X_train.loc[sample_idx]
y_train_sample = y_train.loc[sample_idx]
print(f"  Sampled {len(X_train_sample):,} rows.")

# 2. Random Forest Regressor
print("\nStep 2: Training RandomForestRegressor (on full X_train)...")
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_preds = rf.predict(X_val)

rf_rmse = root_mean_squared_error(y_val, rf_preds)
rf_mae = mean_absolute_error(y_val, rf_preds)
rf_r2 = r2_score(y_val, rf_preds)

print(f"  🌲 RF RMSE: {rf_rmse:.4f}")
print(f"  🌲 RF MAE:  {rf_mae:.4f}")
print(f"  🌲 RF R²:  {rf_r2:.4f}")

# 3. XGBoost Regressor
print("\nStep 3: Training XGBRegressor (on X_train_sample)...")
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train_sample, y_train_sample)
xgb_preds = xgb.predict(X_val)

xgb_rmse = root_mean_squared_error(y_val, xgb_preds)
xgb_mae = mean_absolute_error(y_val, xgb_preds)
xgb_r2 = r2_score(y_val, xgb_preds)
print(f"  ⚡ XGB RMSE: {xgb_rmse:.4f}")
print(f"  ⚡ XGB MAE:  {xgb_mae:.4f}")
print(f"  ⚡ XGB R²:  {xgb_r2:.4f}")
print("\n" + "-" * 40)
print("🏆 MODEL COMPARISON SUMMARY")
print("-" * 40)
print(f"  Random Forest: RMSE = {rf_rmse:.4f} | MAE = {rf_mae:.4f} | R² = {rf_r2:.4f}")
print(f"  XGBoost:       RMSE = {xgb_rmse:.4f} | MAE = {xgb_mae:.4f} | R² = {xgb_r2:.4f}")
print("=" * 60)

🌲 TRAINING ADVANCED MODELS (RF & XGBOOST)
Step 1: Creating training sample for XGBoost...
  Sampled 1,300,000 rows.

Step 2: Training RandomForestRegressor (on full X_train)...
  🌲 RF RMSE: 2.6453
  🌲 RF MAE:  1.9839
  🌲 RF R²:  0.0500

Step 3: Training XGBRegressor (on X_train_sample)...
  ⚡ XGB RMSE: 2.6443
  ⚡ XGB MAE:  1.9824
  ⚡ XGB R²:  0.0507

----------------------------------------
🏆 MODEL COMPARISON SUMMARY
----------------------------------------
  Random Forest: RMSE = 2.6453 | MAE = 1.9839
  XGBoost:       RMSE = 2.6443 | MAE = 1.9824


### Final XGBoost — Full Training Set → Test Set

In [122]:
print("=" * 60)
print("🏆 FINALIZING MODEL & EXPORTING PREDICTIONS")
print("=" * 60)

# 1. Setup Paths & Load Test Data
print("Step 1: Loading strict holdout Test data...")
processed_dir = Path("../data/processed")
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True) 

test = pd.read_csv(processed_dir / "test.csv")

target = "fantasy_points"
structural_cols = ["player_id", "game_id", "date"]

X_test = test.drop(columns=structural_cols + [target])
y_test = test[target]

print(f"  Test shape: X={X_test.shape}, y={y_test.shape}")

# 2. Train Final Model on Complete Training Set
print("\nStep 2: Training final XGBoost on COMPLETE training data (This may take a moment)...")
final_xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

final_xgb.fit(X_train, y_train)
print("  ✅ Training complete.")

# 3. Predict & Evaluate on Unseen Test Data
print("\nStep 3: Predicting on strict holdout timeline (2024+)...")
final_preds = final_xgb.predict(X_test)

final_rmse = root_mean_squared_error(y_test, final_preds)
final_mae = mean_absolute_error(y_test, final_preds)

print("\n" + "-" * 40)
print("🌟 FINAL TEST SET METRICS 🌟")
print("-" * 40)
print(f"  RMSE: {final_rmse:.4f} points")
print(f"  MAE:  {final_mae:.4f} points")
print("-" * 40)

# 4. Save the Model Object
print("\nStep 4: Serializing and saving model architecture...")
model_path = models_dir / "xgb_fantasy_model_v1.joblib"
joblib.dump(final_xgb, model_path)
print(f"  ✅ Model saved to: {model_path}")

# 5. Generate and Save the Enriched Predictions Dataset
print("\nStep 5: Exporting enriched player predictions for Notebook 06...")

# Define the specific columns needed for the recommendation engine
export_cols = [
    "player_id", 
    "player_name", 
    "game_id", 
    "date", 
    "position", 
    "market_value_in_eur",
    "std_fp_last_5", 
    "avg_fp_last_5"
]

# Safely extract requested metadata (handles cases if string cols were dropped during feature prep)
available_cols = [col for col in export_cols if col in test.columns]
predictions_df = test[available_cols].copy()

# Inject the model results
predictions_df["actual_fantasy_points"] = y_test
predictions_df["predicted_fantasy_points"] = final_preds
predictions_df["prediction_error"] = predictions_df["predicted_fantasy_points"] - predictions_df["actual_fantasy_points"]

# Reorder columns to ensure targets sit cleanly next to the metadata
final_col_order = available_cols + ["predicted_fantasy_points", "actual_fantasy_points", "prediction_error"]
predictions_df = predictions_df[final_col_order]

preds_path = processed_dir / "test_predictions.csv"
predictions_df.to_csv(preds_path, index=False)
print(f"  ✅ Predictions saved to: {preds_path}")
print(f"  Preview:")
display(predictions_df.head(3))

print("\n" + "=" * 60)
print("🏁 MODELING PHASE COMPLETE. READY FOR 06_RECOMMENDATION_ENGINE.IPYNB")
print("=" * 60)

🏆 FINALIZING MODEL & EXPORTING PREDICTIONS
Step 1: Loading strict holdout Test data...
  Test shape: X=(349947, 17), y=(349947,)

Step 2: Training final XGBoost on COMPLETE training data (This may take a moment)...
  ✅ Training complete.

Step 3: Predicting on strict holdout timeline (2024+)...

----------------------------------------
🌟 FINAL TEST SET METRICS 🌟
----------------------------------------
  RMSE: 2.6473 points
  MAE:  1.9690 points
----------------------------------------

Step 4: Serializing and saving model architecture...
  ✅ Model saved to: ../models/xgb_fantasy_model_v1.joblib

Step 5: Exporting enriched player predictions for Notebook 06...
  ✅ Predictions saved to: ../data/processed/test_predictions.csv
  Preview:


,player_id,game_id,date,position,market_value_in_eur,std_fp_last_5,avg_fp_last_5,predicted_fantasy_points,actual_fantasy_points,prediction_error
0,434675,4095269,2024-01-01,0,70000000.0,2.509980,3.4,3.824497,6,-2.175503
1,451276,4095269,2024-01-01,3,85000000.0,2.387467,2.8,3.221833,2,1.221833
2,478573,4095269,2024-01-01,3,90000000.0,0.547723,1.4,2.132562,1,1.132562



🏁 MODELING PHASE COMPLETE. READY FOR 06_RECOMMENDATION_ENGINE.IPYNB
